# 01 — Data loading & feature engineering

Connects to the database, loads all tables, derives era classification,
discipline features, play rate, and feature matrices for decks, crypt cards,
and library cards. Saves all artifacts to `data/` for use by the analysis
notebooks (02–04).

**Run this notebook first.** The others will fail if the `data/` artifacts
are absent or stale.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(Path.cwd()))  # for helpers.py

import json
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

from helpers import (
    SEED,
    parse_crypt_disciplines,
    discipline_boolean_matrix,
    parse_library_discipline_names,
    parse_line_start_discipline_tags,
    split_multi,
    cost_to_numeric,
    artifact_path,
    fit_transform,
)
from app.config import settings

np.random.seed(SEED)

In [2]:
engine = create_engine(settings.db.synchrone_url)
prefix = settings.db.table_prefix

with engine.connect() as conn:
    v5_release_date = conn.execute(
        text(f"SELECT release_date FROM {prefix}sets WHERE abbrev = 'V5'")
    ).scalar_one()

print(f"V5 release date: {v5_release_date}")

V5 release date: 2020-11-30


## Load tables

In [3]:
with engine.connect() as conn:
    q_crypt = (
        f'SELECT id, "group", adv, name, clan, type, capacity, path, title,'
        f" disciplines, card_text, banned FROM {prefix}crypt"
    )
    q_library = (
        f"SELECT id, name, type, artist, capacity, pool_cost, blood_cost,"
        f" conviction_cost, clan, path, requirement, flavor_text,"
        f" card_text, discipline, banned, burn_option FROM {prefix}library"
    )
    q_decks = (
        f"SELECT id, name, created_by, description, crypt_count, crypt_min,"
        f" crypt_max, crypt_avg, library_count, tournament_event_id"
        f" FROM {prefix}decks"
    )
    q_tournaments = (
        f"SELECT event_id, name AS tournament_name, location, date_start,"
        f" date_end, players_count, winner, event_url FROM {prefix}tournaments"
    )
    q_deck_crypt = (
        f"SELECT id, deck_id, count, raw_name, raw_grouping,"
        f" crypt_card_id, crypt_card_group, crypt_card_adv"
        f" FROM {prefix}deck_crypt_cards"
    )
    q_deck_library = (
        f"SELECT id, deck_id, section, count, raw_name, library_card_id"
        f" FROM {prefix}deck_library_cards"
    )
    q_crypt_era = (
        f'SELECT ccs.crypt_card_id AS id, ccs.crypt_card_group AS "group",'
        f" ccs.crypt_card_adv AS adv,"
        f" COUNT(*) FILTER (WHERE s.release_date < :cutoff) AS n_pre,"
        f" COUNT(*) FILTER (WHERE s.release_date >= :cutoff) AS n_post"
        f" FROM {prefix}crypt_card_sets ccs"
        f" JOIN {prefix}sets s ON s.id = ccs.set_id"
        f" WHERE s.release_date IS NOT NULL"
        f" GROUP BY ccs.crypt_card_id, ccs.crypt_card_group, ccs.crypt_card_adv"
    )
    q_lib_era = (
        f"SELECT lcs.library_card_id AS id,"
        f" COUNT(*) FILTER (WHERE s.release_date < :cutoff) AS n_pre,"
        f" COUNT(*) FILTER (WHERE s.release_date >= :cutoff) AS n_post"
        f" FROM {prefix}library_card_sets lcs"
        f" JOIN {prefix}sets s ON s.id = lcs.set_id"
        f" WHERE s.release_date IS NOT NULL"
        f" GROUP BY lcs.library_card_id"
    )
    params = {"cutoff": v5_release_date}
    df_crypt = pd.read_sql(text(q_crypt), conn)
    df_library = pd.read_sql(text(q_library), conn)
    df_decks = pd.read_sql(text(q_decks), conn)
    df_tournaments = pd.read_sql(text(q_tournaments), conn)
    df_deck_crypt = pd.read_sql(text(q_deck_crypt), conn)
    df_deck_library = pd.read_sql(text(q_deck_library), conn)
    crypt_era_raw = pd.read_sql(text(q_crypt_era), conn, params=params)
    library_era_raw = pd.read_sql(text(q_lib_era), conn, params=params)

df_decks["id"] = df_decks["id"].astype(str)
df_deck_crypt["deck_id"] = df_deck_crypt["deck_id"].astype(str)
df_deck_library["deck_id"] = df_deck_library["deck_id"].astype(str)

print(
    f"{len(df_crypt)} crypt / {len(df_library)} library / "
    f"{len(df_decks)} decks / {len(df_deck_crypt)} deck-crypt / "
    f"{len(df_deck_library)} deck-library rows"
)

1785 crypt / 2364 library / 854 decks / 6537 deck-crypt / 24784 deck-library rows


## Era classification

In [4]:
def _classify_era(row: pd.Series) -> str:
    if (0 if pd.isna(row["n_pre"]) else row["n_pre"]) > 0:
        return "pre-V5"
    if (0 if pd.isna(row["n_post"]) else row["n_post"]) > 0:
        return "V5+"
    return "unknown"


df_crypt = df_crypt.merge(crypt_era_raw, on=["id", "group", "adv"], how="left")
df_crypt["era"] = df_crypt.apply(_classify_era, axis=1)

df_library = df_library.merge(library_era_raw, on="id", how="left")
df_library["era"] = df_library.apply(_classify_era, axis=1)

print("Crypt era:", df_crypt["era"].value_counts().to_dict())
print("Library era:", df_library["era"].value_counts().to_dict())

Crypt era: {'pre-V5': 1527, 'V5+': 258}
Library era: {'pre-V5': 2212, 'V5+': 152}


## Crypt discipline booleans

In [5]:
def _get_base(t: tuple[set[str], set[str]]) -> set[str]:
    return t[0]


def _get_superior(t: tuple[set[str], set[str]]) -> set[str]:
    return t[1]


parsed_crypt = df_crypt["disciplines"].fillna("").apply(parse_crypt_disciplines)
df_crypt["disc_base"] = parsed_crypt.apply(_get_base)
df_crypt["disc_superior"] = parsed_crypt.apply(_get_superior)

crypt_disc_df = discipline_boolean_matrix(df_crypt["disc_base"], df_crypt["disc_superior"])
disc_cols_base = [c for c in crypt_disc_df.columns if c.islower()]
disc_cols_superior = [c for c in crypt_disc_df.columns if c.isupper()]

violations = sum(
    int((crypt_disc_df[u] & ~crypt_disc_df[l]).sum())
    for l, u in zip(disc_cols_base, disc_cols_superior)
)
print(f"{len(disc_cols_base)} discipline codes, {crypt_disc_df.shape[1]} columns built")
print(f"AUS=>aus violations (must be 0): {violations}")
if len(disc_cols_base) > 40:
    print("WARNING: unexpectedly many codes — check _split_discipline_codes")
    print(df_crypt["disciplines"].dropna().unique()[:10])

36 discipline codes, 72 columns built
AUS=>aus violations (must be 0): 0


## Library discipline features

In [6]:
def _contains(s: set[str], c: str) -> bool:
    return c in s


# Two complementary signals, both derived without guessing a name→code mapping:
#
# 1. `discipline` column → full discipline names ("Auspex", "Animalism & Auspex"…)
#    Used as-is in lowercase; no abbreviation attempted.
#
# 2. Line-start bracket tags in card_text ([dom] / [DOM]…)
#    Tags at the START of a line mark alternate-discipline card versions
#    (structural format, same uppercase=superior convention as crypt).
#    Tags mid-line (e.g. "React with Conviction": "requires Chimerstry [chi]…")
#    describe other cards/minions and are ignored by the regex.
#    → base + superior boolean columns, same AUS⟹aus logic as crypt.

df_library["disc_names"] = df_library["discipline"].fillna("").apply(parse_library_discipline_names)
library_disc_df = pd.DataFrame(
    {
        name: df_library["disc_names"].apply(_contains, args=(name,))
        for name in sorted({n for names in df_library["disc_names"] for n in names})
    },
    index=df_library.index,
)
print(
    f"{library_disc_df.shape[1]} library discipline columns (from `discipline` field):",
    sorted(library_disc_df.columns),
)
if library_disc_df.shape[1] > 40:
    print("WARNING: unexpectedly many names — check parse_library_discipline_names")

39 library discipline columns (from `discipline` field): ['abombwe', 'animalism', 'auspex', 'celerity', 'chimerstry', 'daimoinon', 'defense', 'dementation', 'dominate', 'flight', 'fortitude', 'innocence', 'judgment', 'maleficia', 'martyrdom', 'melpominee', 'mytherceria', 'necromancy', 'obeah', 'obfuscate', 'oblivion', 'obtenebration', 'potence', 'presence', 'protean', 'quietus', 'redemption', 'sanguinus', 'serpentis', 'spiritus', 'striga', 'temporis', 'thanatosis', 'thaumaturgy', 'valeren', 'vengeance', 'vicissitude', 'visceratika', 'vision']


In [7]:
# Line-start bracket tags — verification + parsing
parsed_tags = df_library["card_text"].fillna("").apply(parse_line_start_discipline_tags)
df_library["tag_base"] = parsed_tags.apply(_get_base)
df_library["tag_superior"] = parsed_tags.apply(_get_superior)

n_with_tags = (df_library["tag_base"].apply(len) > 0).sum()
all_tag_codes = sorted({c for s in df_library["tag_base"] for c in s})
print(f"{n_with_tags} library cards have line-start discipline tags.")
print(f"Distinct codes found: {all_tag_codes}")

# Verify AUS⟹aus holds for these tags too
library_tag_df = discipline_boolean_matrix(df_library["tag_base"], df_library["tag_superior"])
tag_cols_base = [c for c in library_tag_df.columns if c.islower()]
tag_cols_superior = [c for c in library_tag_df.columns if c.isupper()]
violations = sum(
    int((library_tag_df[u] & ~library_tag_df[l]).sum())
    for l, u in zip(tag_cols_base, tag_cols_superior)
)
print(f"AUS=>aus violations (must be 0): {violations}")
print(f"{library_tag_df.shape[1]} tag boolean columns built.")
library_tag_df.head()

819 library cards have line-start discipline tags.
Distinct codes found: ['abo', 'ani', 'aus', 'cel', 'chi', 'dai', 'dem', 'dom', 'for', 'mal', 'mel', 'myt', 'nec', 'obe', 'obf', 'obl', 'obt', 'pot', 'pre', 'pro', 'qui', 'san', 'ser', 'spi', 'str', 'tem', 'tha', 'thn', 'val', 'vic', 'vis']
AUS=>aus violations (must be 0): 0
62 tag boolean columns built.


,abo,ABO,ani,ANI,aus,AUS,cel,CEL,chi,CHI,...,tha,THA,thn,THN,val,VAL,vic,VIC,vis,VIS
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Library card type (multi-hot)

In [8]:
df_library["type_list"] = df_library["type"].fillna("").apply(split_multi)
mlb_type = MultiLabelBinarizer()
type_matrix = fit_transform(mlb_type, df_library["type_list"])

library_type_df = pd.DataFrame(
    type_matrix,
    columns=[f"type_{t}" for t in mlb_type.classes_],
    index=df_library.index,
)
print(f"Library types: {sorted(mlb_type.classes_)}")

Library types: ['Action', 'Action Modifier', 'Ally', 'Combat', 'Conviction', 'Equipment', 'Event', 'Master', 'Political Action', 'Power', 'Reaction', 'Retainer']


## Play rate

In [9]:
total_decks = len(df_decks)
df_crypt["id"] = df_crypt["id"].astype(int)
df_library["id"] = df_library["id"].astype(int)

crypt_play = (
    df_deck_crypt[df_deck_crypt["crypt_card_id"].notna()]
    .groupby(["crypt_card_id", "crypt_card_group", "crypt_card_adv"])
    .agg(n_decks_played=("deck_id", "nunique"), total_copies=("count", "sum"))
    .reset_index()
    .rename(
        columns={
            "crypt_card_id": "id",
            "crypt_card_group": "group",
            "crypt_card_adv": "adv",
        }
    )
)
crypt_play["id"] = crypt_play["id"].astype(int)
df_crypt = df_crypt.merge(crypt_play, on=["id", "group", "adv"], how="left")
df_crypt[["n_decks_played", "total_copies"]] = df_crypt[["n_decks_played", "total_copies"]].fillna(
    0
)
df_crypt["play_rate"] = df_crypt["n_decks_played"] / total_decks if total_decks else 0.0
df_crypt["played"] = df_crypt["play_rate"] > 0

library_play = (
    df_deck_library[df_deck_library["library_card_id"].notna()]
    .groupby("library_card_id")
    .agg(n_decks_played=("deck_id", "nunique"), total_copies=("count", "sum"))
    .reset_index()
    .rename(columns={"library_card_id": "id"})
)
library_play["id"] = library_play["id"].astype(int)
df_library = df_library.merge(library_play, on="id", how="left")
df_library[["n_decks_played", "total_copies"]] = df_library[
    ["n_decks_played", "total_copies"]
].fillna(0)
df_library["play_rate"] = df_library["n_decks_played"] / total_decks if total_decks else 0.0
df_library["played"] = df_library["play_rate"] > 0

print(f"Crypt played: {int(df_crypt['played'].sum())} / {len(df_crypt)}")
print(f"Library played: {int(df_library['played'].sum())} / {len(df_library)}")

Crypt played: 987 / 1785
Library played: 1226 / 2364


## Feature matrices

In [10]:
# --- Crypt ---
cat_cols_crypt = ["clan", "title", "era"]
df_crypt[cat_cols_crypt] = df_crypt[cat_cols_crypt].fillna("")
ohe_crypt = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
cat_matrix_crypt = fit_transform(ohe_crypt, df_crypt[cat_cols_crypt])
cat_df_crypt = pd.DataFrame(
    cat_matrix_crypt,
    columns=ohe_crypt.get_feature_names_out(cat_cols_crypt),
    index=df_crypt.index,
)
num_df_crypt = pd.DataFrame(
    {
        "capacity": df_crypt["capacity"],
        "adv": df_crypt["adv"].astype(int),
        "banned": df_crypt["banned"].astype(int),
        "n_disciplines": df_crypt["disc_base"].apply(len),
    },
    index=df_crypt.index,
)

X_crypt = pd.concat([num_df_crypt, crypt_disc_df.astype(int), cat_df_crypt], axis=1)
X_crypt_scaled = fit_transform(StandardScaler(), X_crypt)
print(f"Crypt feature matrix: {X_crypt.shape}")

Crypt feature matrix: (1785, 141)


In [11]:
# --- Library ---
cat_cols_lib = ["clan", "era"]
df_library[cat_cols_lib] = df_library[cat_cols_lib].fillna("")
ohe_lib = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
cat_matrix_lib = fit_transform(ohe_lib, df_library[cat_cols_lib])
cat_df_lib = pd.DataFrame(
    cat_matrix_lib,
    columns=ohe_lib.get_feature_names_out(cat_cols_lib),
    index=df_library.index,
)

for col in ["pool_cost", "blood_cost", "conviction_cost"]:
    df_library[f"{col}_num"] = df_library[col].apply(cost_to_numeric)
    df_library[f"{col}_is_x"] = (df_library[col].astype(str).str.upper() == "X").astype(int)

num_df_lib = pd.DataFrame(
    {
        "pool_cost_num": df_library["pool_cost_num"],
        "blood_cost_num": df_library["blood_cost_num"],
        "conviction_cost_num": df_library["conviction_cost_num"],
        "pool_cost_is_x": df_library["pool_cost_is_x"],
        "blood_cost_is_x": df_library["blood_cost_is_x"],
        "conviction_cost_is_x": df_library["conviction_cost_is_x"],
        "burn_option": df_library["burn_option"].astype(int),
        "banned": df_library["banned"].astype(int),
        "text_length": df_library["card_text"].fillna("").str.len(),
        "n_disciplines": df_library["disc_names"].apply(len),
    },
    index=df_library.index,
)

tfidf = TfidfVectorizer(max_features=300, stop_words="english", min_df=3)
text_matrix = fit_transform(tfidf, df_library["card_text"].fillna(""))
n_svd = max(2, min(20, text_matrix.shape[1] - 1))
svd = TruncatedSVD(n_components=n_svd, random_state=SEED)
text_svd = svd.fit_transform(text_matrix)
text_svd_df = pd.DataFrame(
    text_svd, columns=[f"text_svd_{i}" for i in range(n_svd)], index=df_library.index
)
print(
    f"TF-IDF + SVD ({n_svd} components): "
    f"{svd.explained_variance_ratio_.sum():.1%} variance explained"
)

X_library = pd.concat(
    [
        num_df_lib,
        library_type_df.astype(int),
        library_disc_df.astype(int),
        library_tag_df.astype(int),
        cat_df_lib,
        text_svd_df,
    ],
    axis=1,
).fillna(0)
X_library_scaled = fit_transform(StandardScaler(), X_library)
print(f"Library feature matrix: {X_library.shape}")

TF-IDF + SVD (20 components): 31.2% variance explained
Library feature matrix: (2364, 191)


In [12]:
# --- Deck x card matrix ---
dc = df_deck_crypt[df_deck_crypt["crypt_card_id"].notna()].copy()
dc["card_key"] = (
    "crypt_"
    + dc["crypt_card_id"].astype(int).astype(str)
    + "_"
    + dc["crypt_card_group"].astype(str)
    + "_"
    + dc["crypt_card_adv"].astype(str)
)
crypt_matrix = dc.pivot_table(
    index="deck_id",
    columns="card_key",
    values="count",
    aggfunc="sum",
    fill_value=0,
)

dl = df_deck_library[df_deck_library["library_card_id"].notna()].copy()
dl["card_key"] = "lib_" + dl["library_card_id"].astype(int).astype(str)
library_matrix = dl.pivot_table(
    index="deck_id",
    columns="card_key",
    values="count",
    aggfunc="sum",
    fill_value=0,
)

deck_matrix = pd.concat([crypt_matrix, library_matrix], axis=1).reindex(df_decks["id"]).fillna(0)
svd_deck = TruncatedSVD(n_components=50, random_state=SEED)
X_deck_reduced = svd_deck.fit_transform(deck_matrix.values)
print(f"Deck x card matrix: {deck_matrix.shape}")
print(
    f"Unresolved: {len(df_deck_crypt) - len(dc)} crypt, "
    f"{len(df_deck_library) - len(dl)} library rows"
)
print(
    f"TruncatedSVD 50 components → "
    f"{svd_deck.explained_variance_ratio_.sum():.1%} variance explained"
)

Deck x card matrix: (854, 2213)
Unresolved: 0 crypt, 35 library rows
TruncatedSVD 50 components → 74.8% variance explained


## Save artifacts

In [13]:
artifact_path("").mkdir(exist_ok=True)

# DataFrames
df_crypt.drop(columns=["disc_base", "disc_superior", "disc_names"], errors="ignore").to_parquet(
    artifact_path("df_crypt.parquet")
)
df_library.drop(columns=["disc_names", "type_list"], errors="ignore").to_parquet(
    artifact_path("df_library.parquet")
)
df_decks.to_parquet(artifact_path("df_decks.parquet"))
df_deck_crypt.to_parquet(artifact_path("df_deck_crypt.parquet"))
df_deck_library.to_parquet(artifact_path("df_deck_library.parquet"))

# Boolean matrices
crypt_disc_df.to_parquet(artifact_path("crypt_disc_df.parquet"))
library_disc_df.to_parquet(artifact_path("library_disc_df.parquet"))
library_tag_df.to_parquet(artifact_path("library_tag_df.parquet"))
library_type_df.to_parquet(artifact_path("library_type_df.parquet"))
deck_matrix.to_parquet(artifact_path("deck_matrix.parquet"))

# Scaled arrays + column names
np.save(artifact_path("X_crypt_scaled.npy"), X_crypt_scaled)
np.save(artifact_path("X_library_scaled.npy"), X_library_scaled)
np.save(artifact_path("X_deck_scaled.npy"), X_deck_scaled)

artifact_path("X_crypt_cols.json").write_text(json.dumps(list(X_crypt.columns)), encoding="utf-8")
artifact_path("X_library_cols.json").write_text(
    json.dumps(list(X_library.columns)), encoding="utf-8"
)

# card_key → name lookup
card_key_to_name: dict[str, str] = {}
for _, r in df_crypt.iterrows():
    card_key_to_name[f"crypt_{int(r['id'])}_{r['group']}_{r['adv']}"] = r["name"]
for _, r in df_library.iterrows():
    card_key_to_name[f"lib_{int(r['id'])}"] = r["name"]
artifact_path("card_key_to_name.json").write_text(json.dumps(card_key_to_name), encoding="utf-8")

print("All artifacts saved to data/")

NameError: name 'X_deck_scaled' is not defined